In [1]:
import geopandas as gpd
from sentle import sentle
import os
from dotenv import load_dotenv
from pathlib import Path
import torch
import concurrent.futures
import logging
import time
import sys
import shutil

# Configure logging
logging.basicConfig(
    level=logging.INFO,  # Set the desired logging level
    format='%(asctime)s - %(levelname)s - %(message)s',
    handlers=[
        logging.StreamHandler(),  # Logs to the console
        logging.FileHandler('sentinel_processing_4.log')  # Logs to a file
    ]
)
logger = logging.getLogger()


In [5]:
def load_sentle(grid_path, idx, res):
    """
    Load Sentinel data using the Sentle library for a given grid.
    """
    try:
        intersected_gdf_equi7 = gpd.read_file(grid_path)
        id_n = idx + 1
        bounds = intersected_gdf_equi7[idx:id_n].geometry.iloc[0].bounds
        bound_left = int(bounds[0])
        bound_bottom = int(bounds[1])
        bound_right = int(bounds[2])
        bound_top = int(bounds[3])
        equi7_crs = intersected_gdf_equi7.crs
        logger.info(f"Resolution: {res}")
        print(f"Minicube: {idx}, bound_left = {bound_left}, bound_bottom = {bound_bottom}, bound_right = {bound_right}, bound_top = {bound_top}\nCRS = {equi7_crs} ")

        # da = sentle.process(
        #     target_crs=equi7_crs,
        #     bound_left=bound_left,
        #     bound_bottom=bound_bottom,
        #     bound_right=bound_right,
        #     bound_top=bound_top,
        #     datetime=f"2015-01-01/2024-07-31",
        #     target_resolution=res,
        #     dask_scheduler_port=10012,
        #     dask_dashboard_address='127.0.0.1:37382',
        #     S2_mask_snow=True,
        #     S2_cloud_classification=True,
        #     S2_cloud_classification_device="cuda",
        #     S1_assets=["vv", "vh"],
        #     S2_apply_snow_mask=True,
        #     S2_apply_cloud_mask=True,
        #     time_composite_freq="7d",
        #     # NOTE clemens: this can be set to 40
        #     num_workers=40,
        # )
        # return da
    except Exception as e:
        logger.error(f"Error in load_sentle function for index {idx}: {e}")
        raise

def process_and_save(grid_path, idx, res, year, output_zarr_path):
    """
    Process and save Sentinel data for a given index.
    """
    try:
        logger.info(f"> Load the Minicube {idx} ...")
        da = load_sentle(grid_path=grid_path, idx=idx, res=res)
        logger.info(f"> Save the Minicube {idx} ...")
        #output_zarr_path = f"{os.getenv('SENTINEL2_MINICUBES')}/{idx}_{res}_512_20152024_equi7_NA.zarr"
        sentle.save_as_zarr(da, path=output_zarr_path)
        logger.info(f"> Successfully saved the Minicube {idx} at {output_zarr_path} ...")
    except Exception as e:
        logger.error(f"An error occurred for index {idx}: {e}")


In [11]:
idx = 856

# Load the Environment variables
env_path = Path('/net/projects/forexd/WP1/02_ImprovedLabels/Scripts/ForExD-WP1-P1/environment/.env')
load_dotenv(dotenv_path=env_path)

path = f"{os.getenv('DATA')}/{idx}_10_512_20152024_equi7_NA.zarr"


# Check if the folder exists, then delete it along with all its subfolders
if os.path.exists(path):
    print(f"Deleting folder and subfolders at: {path}")
    shutil.rmtree(path)
else:
    print(f"Folder does not exist: {path}")

# Set CUDA environment
os.environ["CUDA_VISIBLE_DEVICES"] = "2"
logger.info(f"> Available CUDA devices: {torch.cuda.device_count()}")

# Ensure the 'REGION' environment variable is set
region = os.getenv('REGION')
if region is None:
    raise ValueError("The 'REGION' environment variable is not set. Please ensure it is defined in the .env file.")

print(f"Working on USDA Region {region} ...")
region_id=str(region).zfill(2)

# Set resolution of the grid that  i want to load
resolution = 10
pixel_size = 512
# Path to grid_file
grid_path = f"{os.getenv('EQUI7_GRIDS')}/grid_equi7_{resolution}_{pixel_size}_region_{region_id}_intersetion.shp"
process_and_save(grid_path, idx, resolution, year, path)

2024-10-02 08:43:34,538 - INFO - > Available CUDA devices: 1
2024-10-02 08:43:34,539 - INFO - > Load the Minicube 856 ...
2024-10-02 08:43:34,561 - INFO - Resolution: 10
2024-10-02 08:43:34,563 - INFO - > Save the Minicube 856 ...
2024-10-02 08:43:34,565 - ERROR - An error occurred for index 856: 'NoneType' object has no attribute 'rename'


Folder does not exist: /net/projects/forexd/WP1/Data//856_10_512_20152024_equi7_NA.zarr
Working on USDA Region 8 ...
Minicube: 856, bound_left = 9594880, bound_bottom = 3092480, bound_right = 9600000, bound_top = 3097600
CRS = EPSG:27705 


In [25]:
import os
import shutil
from dotenv import load_dotenv
from pathlib import Path
import torch
import logging

# Define a logger
logger = logging.getLogger(__name__)
logger.setLevel(logging.INFO)

# Define the indices and the year range
indices = [114, 115, 125, 126, 242, 479, 856]
start_year = 2015
end_year = 2024

# Loop through each index
for idx in indices:
    # Loop through each year from 2015 to 2024
    for year in range(start_year, end_year + 1):
        try:
            # Print the current index and year
            print(f"Processing idx={idx} for year={year}")
            
            # Load the Environment variables
            env_path = Path('/net/projects/forexd/WP1/02_ImprovedLabels/Scripts/ForExD-WP1-P1/environment/.env')
            load_dotenv(dotenv_path=env_path)

            path = f"{os.getenv('DATA')}/{idx}_{year}_10_512_20152024_equi7_NA.zarr"

            # Check if the folder exists, then delete it along with all its subfolders
            if os.path.exists(path):
                print(f"Deleting folder and subfolders at: {path}")
                shutil.rmtree(path)
            else:
                print(f"Folder does not exist: {path}")

            # Set CUDA environment
            os.environ["CUDA_VISIBLE_DEVICES"] = "2"
            logger.info(f"> Available CUDA devices: {torch.cuda.device_count()}")

            # Ensure the 'REGION' environment variable is set
            region = os.getenv('REGION')
            if region is None:
                raise ValueError("The 'REGION' environment variable is not set. Please ensure it is defined in the .env file.")

            print(f"Working on USDA Region {region} ...")
            region_id = str(region).zfill(2)

            # Set resolution of the grid
            resolution = 10
            pixel_size = 512
            # Path to grid_file
            grid_path = f"{os.getenv('EQUI7_GRIDS')}/grid_equi7_{resolution}_{pixel_size}_region_{region_id}_intersetion.shp"
            
            # Process the data and save (your function)
            process_and_save(grid_path, idx, resolution, year, path)

        except Exception as e:
            # Catch any errors, log them, and continue with the next loop iteration
            logger.error(f"Error processing idx={idx}, year={year}: {e}")
            print(f"Error occurred with idx={idx} and year={year}: {e}")
            continue


2024-10-01 10:02:19,591 - INFO - > Available CUDA devices: 1
2024-10-01 10:02:19,592 - INFO - > Load the Minicube 114 ...
2024-10-01 10:02:19,614 - INFO - Resolution: 10


Processing idx=114 for year=2015
Folder does not exist: /net/projects/forexd/WP1/Data//114_2015_10_512_20152024_equi7_NA.zarr
Working on USDA Region 8 ...


2024-10-01 10:02:22,538 - INFO - > Save the Minicube 114 ...
2024-10-01 10:02:46,292 - INFO - > Successfully saved the Minicube 114 at /net/projects/forexd/WP1/Data//114_2015_10_512_20152024_equi7_NA.zarr ...
2024-10-01 10:02:46,301 - INFO - > Available CUDA devices: 1
2024-10-01 10:02:46,302 - INFO - > Load the Minicube 114 ...
2024-10-01 10:02:46,328 - INFO - Resolution: 10


Processing idx=114 for year=2016
Folder does not exist: /net/projects/forexd/WP1/Data//114_2016_10_512_20152024_equi7_NA.zarr
Working on USDA Region 8 ...


2024-10-01 10:02:47,785 - INFO - > Save the Minicube 114 ...
2024-10-01 10:03:40,967 - INFO - > Successfully saved the Minicube 114 at /net/projects/forexd/WP1/Data//114_2016_10_512_20152024_equi7_NA.zarr ...
2024-10-01 10:03:40,979 - INFO - > Available CUDA devices: 1
2024-10-01 10:03:40,980 - INFO - > Load the Minicube 114 ...
2024-10-01 10:03:41,004 - INFO - Resolution: 10


...

2024-10-01 10:46:55,827 - INFO - > Save the Minicube 856 ...
2024-10-01 10:47:44,221 - INFO - > Successfully saved the Minicube 856 at /net/projects/forexd/WP1/Data//856_2022_10_512_20152024_equi7_NA.zarr ...
2024-10-01 10:47:44,244 - INFO - > Available CUDA devices: 1
2024-10-01 10:47:44,245 - INFO - > Load the Minicube 856 ...
2024-10-01 10:47:44,280 - INFO - Resolution: 10


Processing idx=856 for year=2023
Folder does not exist: /net/projects/forexd/WP1/Data//856_2023_10_512_20152024_equi7_NA.zarr
Working on USDA Region 8 ...


2024-10-01 10:47:46,672 - INFO - > Save the Minicube 856 ...
2024-10-01 10:48:09,062 - distributed.worker - ERROR - Compute Failed
Key:       ('process_ptile-this-store-map-a59daf6a505f970edd2ffbabbfb76020', 51, 0, 0, 0)
State:     executing
Function:  execute_task
args:      ((<function store_chunk at 0x7b2ee39c07c0>, (<built-in function getitem>, (<function map_blocks.<locals>._wrapper at 0x7b2e1a49a700>, <function process_ptile at 0x7b2e5cfe4180>, [(<class 'xarray.core.dataset.Dataset'>, (<class 'dict'>, [['full_like-e1c196bad4f2a81b5adbd1369d67f95e', (('time', 'band', 'y', 'x'), array([[[[nan, nan, nan, ..., nan, nan, nan],
         [nan, nan, nan, ..., nan, nan, nan],
         [nan, nan, nan, ..., nan, nan, nan],
         ...,
         [nan, nan, nan, ..., nan, nan, nan],
         [nan, nan, nan, ..., nan, nan, nan],
         [nan, nan, nan, ..., nan, nan, nan]],

        [[nan, nan, nan, ..., nan, nan, nan],
         [nan, nan, nan, ..., nan, nan, nan],
         [nan, nan, nan, .

Processing idx=856 for year=2024
Folder does not exist: /net/projects/forexd/WP1/Data//856_2024_10_512_20152024_equi7_NA.zarr
Working on USDA Region 8 ...


2024-10-01 10:48:11,147 - INFO - > Save the Minicube 856 ...
2024-10-01 10:49:04,070 - INFO - > Successfully saved the Minicube 856 at /net/projects/forexd/WP1/Data//856_2024_10_512_20152024_equi7_NA.zarr ...


In [11]:
process_and_save(grid_path, idx, resolution)

2024-10-01 09:11:21,641 - INFO - > Load the Minicube 114 ...
2024-10-01 09:11:21,665 - INFO - Resolution: 10
2024-10-01 09:11:23,923 - INFO - > Save the Minicube 114 ...
2024-10-01 09:12:36,716 - INFO - > Successfully saved the Minicube 114 at /net/projects/forexd/WP1/Data/S2_Cubes_IDS_R8//114_10_512_20152024_equi7_NA.zarr ...


In [5]:
logger.info(f"> Load the Minicube {idx} ...")
da = load_sentle(grid_path=grid_path, idx=idx, res=resolution)
da

2024-09-30 15:38:24,125 - INFO - > Load the Minicube 125 ...
2024-09-30 15:38:24,145 - INFO - Resolution: 10
2024-09-30 15:38:24,761 - distributed.protocol.core - INFO - Failed to serialize (can not serialize 'GeoDataFrame' object); falling back to pickle. Be aware that this may degrade performance.


<xarray.DataArray 'full_like-00f727bc1bab33b5a49e90fed72b4ce1' (time: 428,
                                                                band: 14,
                                                                y: 512, x: 512)> Size: 6GB
dask.array<<this-array>-process_ptile, shape=(428, 14, 512, 512), dtype=float32, chunksize=(1, 12, 512, 512), chunktype=numpy.ndarray>
Coordinates:
  * band     (band) <U3 168B 'B01' 'B02' 'B03' 'B04' ... 'B11' 'B12' 'vv' 'vh'
  * x        (x) float32 2kB 8.709e+06 8.709e+06 ... 8.714e+06 8.714e+06
  * y        (y) float32 2kB 2.632e+06 2.632e+06 ... 2.627e+06 2.627e+06
  * time     (time) datetime64[ns] 3kB 2024-08-01 2024-07-25 ... 2015-03-12

In [6]:
da.isel(x=100,y=100).sel(band="B04").plot()

TypeError: can't multiply sequence by non-int of type 'NoneType'

In [ ]:
logger.info(f"> Save the Minicube {idx} ...")
output_zarr_path = f"{os.getenv('SENTINEL2_MINICUBES')}/{idx}_{resolution}_512_20152024_equi7_NA.zarr"
sentle.save_as_zarr(da, path=output_zarr_path)
logger.info(f"> Successfully saved the Minicube {idx} at {output_zarr_path} ...")
logger.error(f"An error occurred for index {idx}: {e}")